In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/imdb-movie-reviews/IMDB Dataset.csv


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vishakhdapat/imdb-movie-reviews")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/imdb-movie-reviews


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
import pandas as pd
df=pd.read_csv('/kaggle/input/imdb-movie-reviews/IMDB Dataset.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [3]:
df['review'] = df['review'].str.replace('<br />', '', regex=False)

In [4]:
X = df['review']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# Create BoW features
bow_vectorizer = CountVectorizer(max_features=1000) 
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# Train Naive Bayes
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)

# Evaluate
y_pred = nb_bow.predict(X_test_bow)
print("Naive Bayes with BoW Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Naive Bayes with BoW Accuracy: 0.8168
              precision    recall  f1-score   support

    negative       0.81      0.82      0.82      4961
    positive       0.82      0.81      0.82      5039

    accuracy                           0.82     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.82      0.82      0.82     10000



In [8]:
# Train Logistic Regression
lr_bow = LogisticRegression(max_iter=1000)
lr_bow.fit(X_train_bow, y_train)

# Evaluate
y_pred = lr_bow.predict(X_test_bow)
print("Logistic Regression with BoW Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Logistic Regression with BoW Accuracy: 0.8687
              precision    recall  f1-score   support

    negative       0.87      0.86      0.87      4961
    positive       0.86      0.88      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



TF-IDF

In [9]:
tfidf_vectorizer = TfidfVectorizer(max_features=2000)  # Limit to top 5000 words
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [16]:
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

y_pred = nb_tfidf.predict(X_test_tfidf)
print("Naive Bayes with TF-IDF Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Naive Bayes with TF-IDF Accuracy: 0.8517
              precision    recall  f1-score   support

    negative       0.85      0.85      0.85      4961
    positive       0.85      0.85      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



In [10]:
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)

y_pred1 = lr_tfidf.predict(X_test_tfidf)
print("Logistic Regression with TF-IDF Accuracy:", accuracy_score(y_test, y_pred1))
print(classification_report(y_test, y_pred1))

Logistic Regression with TF-IDF Accuracy: 0.886
              precision    recall  f1-score   support

    negative       0.89      0.88      0.88      4961
    positive       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [14]:
import numpy as np

# Identify cases where TF-IDF model is correct and incorrect
correct_tfidf = (y_pred1 == y_test)  # Correct predictions
incorrect_tfidf = (y_pred1 != y_test)  # Misclassified cases

# Find indices for correct and incorrect predictions
indices_correct_tfidf = np.where(correct_tfidf)[0]
indices_incorrect_tfidf = np.where(incorrect_tfidf)[0]

def is_valid_review(text, min_words=50, max_words=250):
    return min_words <= len(text.split()) <= max_words 

# Filter cases based on word count and <br /> removal
filtered_correct_tfidf = [i for i in indices_correct_tfidf if is_valid_review(X_test.iloc[i])]
filtered_incorrect_tfidf = [i for i in indices_incorrect_tfidf if is_valid_review(X_test.iloc[i])]

# Print up to 3 correctly classified cases
print("Examples where TF-IDF + Logistic Regression predicts correctly:\n")
for i in filtered_correct_tfidf[:3]:  # Limit to 3 cases
    print(f"Review: {X_test.iloc[i]}")
    print(f"Actual Label: {y_test.iloc[i]}")
    print(f"TF-IDF Prediction: {y_pred1[i]}")
    print("-" * 80)



Examples where TF-IDF + Logistic Regression predicts correctly:

Review: For once a story of hope highlighted over the tragic reality our youth face. Favela Rising draws one into a scary, unsafe and unfair world and shows through beautiful color and moving music how one man and his dedicated friends choose not to accept that world and change it through action and art. An entertaining, interesting, emotional, aesthetically beautiful film. I showed this film to numerous high school students as well who all live in neighborhoods with poverty and and gun violence and they were enamored with Anderson, the protagonist. I recommend this film to all ages over 13 (due to subtitles and some images of death) from all backgrounds.
Actual Label: positive
TF-IDF Prediction: positive
--------------------------------------------------------------------------------
Review: jeez, this was immensely boring. the leading man (Christian Schoyen) has got to be the worst actor i have ever seen. and another th

In [15]:
# Print up to 3 incorrectly classified cases
print("\nExamples where TF-IDF + Logistic Regression fails:\n")
for i in filtered_incorrect_tfidf[:3]:  # Limit to 3 cases
    print(f"Review: {X_test.iloc[i]}")
    print(f"Actual Label: {y_test.iloc[i]}")
    print(f"TF-IDF Prediction: {y_pred1[i]}")
    print("-" * 80)



Examples where TF-IDF + Logistic Regression fails:

Review: I really liked this Summerslam due to the look of the arena, the curtains and just the look overall was interesting to me for some reason. Anyways, this could have been one of the best Summerslam's ever if the WWF didn't have Lex Luger in the main event against Yokozuna, now for it's time it was ok to have a huge fat man vs a strong man but I'm glad times have changed. It was a terrible main event just like every match Luger is in is terrible. Other matches on the card were Razor Ramon vs Ted Dibiase, Steiner Brothers vs Heavenly Bodies, Shawn Michaels vs Curt Hening, this was the event where Shawn named his big monster of a body guard Diesel, IRS vs 1-2-3 Kid, Bret Hart first takes on Doink then takes on Jerry Lawler and stuff with the Harts and Lawler was always very interesting, then Ludvig Borga destroyed Marty Jannetty, Undertaker took on Giant Gonzalez in another terrible match, The Smoking Gunns and Tatanka took on Bam

In [18]:
rf_tfidf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_tfidf.fit(X_train_tfidf, y_train)

y_pred = rf_tfidf.predict(X_test_tfidf)
print("Random Forest with TF-IDF Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Random Forest with TF-IDF Accuracy: 0.8515
              precision    recall  f1-score   support

    negative       0.84      0.86      0.85      4961
    positive       0.86      0.84      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



In [19]:
svm_tfidf = SVC(kernel='linear')
svm_tfidf.fit(X_train_tfidf, y_train)

y_pred = svm_tfidf.predict(X_test_tfidf)
print("SVM with TF-IDF Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

SVM with TF-IDF Accuracy: 0.8941
              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4961
    positive       0.89      0.90      0.90      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [6]:
# Get misclassified examples
misclassified = X_test[y_test != y_pred1]

# Create a DataFrame with review, true label, and predicted label
misclassified_df = pd.DataFrame({
    'review': X_test[y_test != y_pred1],
    'true_sentiment': y_test[y_test != y_pred1],
    'predicted_sentiment': y_pred1[y_test != y_pred1]
})

# Add the probability scores for each prediction
probabilities = lr_tfidf.predict_proba(X_test_tfidf)[y_test != y_pred1]
misclassified_df['probability_negative'] = probabilities[:, 0]  # Assuming 0 is negative
misclassified_df['probability_positive'] = probabilities[:, 1]  # Assuming 1 is positive

# Save to CSV
misclassified_df.to_csv('misclassified_reviews_lr_tfidf1.csv', index=False)

print(f"Saved {len(misclassified_df)} misclassified examples to CSV.")

Saved 1041 misclassified examples to CSV.


In [7]:
# Filter misclassified cases
false_positives = misclassified_df[
    (misclassified_df['true_sentiment'] == 'positive') & (misclassified_df['predicted_sentiment'] == 'negative')
]

false_negatives = misclassified_df[
    (misclassified_df['true_sentiment'] == 'negative') & (misclassified_df['predicted_sentiment'] == 'positive')
]

# Save to CSV
false_positives.to_csv('false_positives_tf-idf.csv', index=False)
false_negatives.to_csv('false_negatives_tf-idf.csv', index=False)

print(f"Saved {len(false_positives)} false positives and {len(false_negatives)} false negatives to CSV.")


Saved 460 false positives and 581 false negatives to CSV.


In [23]:
# Get misclassified examples and create initial DataFrame
misclassified_df = pd.DataFrame({
    'review': X_test[y_test != y_pred1],
    'true_sentiment': y_test[y_test != y_pred1],
    'predicted_sentiment': y_pred1[y_test != y_pred1]
})

# Add probability scores
probabilities = lr_tfidf.predict_proba(X_test_tfidf)[y_test != y_pred1]
misclassified_df['probability_negative'] = probabilities[:, 0]
misclassified_df['probability_positive'] = probabilities[:, 1]

# --- ADD YOUR NEW FEATURES HERE ---
# Add text length as a feature
misclassified_df['review_length'] = misclassified_df['review'].apply(len)

# Add word count
misclassified_df['word_count'] = misclassified_df['review'].apply(lambda x: len(x.split()))

# Add confidence score
misclassified_df['confidence'] = misclassified_df.apply(
    lambda row: row['probability_negative'] if row['predicted_sentiment'] == 'negative' 
    else row['probability_positive'], axis=1)
# ---------------------------------

# Save to CSV
misclassified_df.to_csv('misclassified_reviews_lr_tfidf2.csv', index=False)

print(f"Saved {len(misclassified_df)} misclassified examples to CSV.")

Saved 1041 misclassified examples to CSV.


WORD2VEC ALONG WITH CLASSIFIERS

In [8]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

# Tokenize reviews
tokenized_reviews = [word_tokenize(review.lower()) for review in X_train]

# Train Word2Vec model
w2v_model = Word2Vec(tokenized_reviews,
                    vector_size=300,  # embedding dimension
                    window=5,         # context window size
                    min_count=5,      # minimum word frequency
                    workers=4,        # parallel threads
                    epochs=10)       # training iterations

# Save the model
w2v_model.save("word2vec_imdb.model")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [9]:
import numpy as np

def average_word_vectors(review, model, vocabulary, num_features):
    feature_vector = np.zeros((num_features,), dtype="float64")
    nwords = 0
    
    for word in word_tokenize(review.lower()):
        if word in vocabulary:
            feature_vector = np.add(feature_vector, model.wv[word])
            nwords += 1
    
    if nwords:
        feature_vector = np.divide(feature_vector, nwords)
        
    return feature_vector

# Get vocabulary
vocabulary = set(w2v_model.wv.index_to_key)

# Create averaged feature vectors
X_train_w2v = np.array([average_word_vectors(review, w2v_model, vocabulary, 300) 
                       for review in X_train])
X_test_w2v = np.array([average_word_vectors(review, w2v_model, vocabulary, 300) 
                      for review in X_test])

In [10]:
lr_w2v = LogisticRegression(max_iter=1000)
lr_w2v.fit(X_train_w2v, y_train)

y_pred = lr_w2v.predict(X_test_w2v)
print("Logistic Regression with Word2Vec Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Logistic Regression with Word2Vec Accuracy: 0.8742
              precision    recall  f1-score   support

    negative       0.88      0.87      0.87      4961
    positive       0.87      0.88      0.88      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [11]:
# Identify cases where TF-IDF fails but Word2Vec succeeds
# incorrect_tfidf = (y_pred1 != y_test)  # TF-IDF misclassified
# correct_w2v = (y_pred == y_test)  # Word2Vec correctly classified

# # Find indices where TF-IDF fails and Word2Vec succeeds
# indices = np.where(incorrect_tfidf & correct_w2v)[0]

# # Print up to 3 cases where Word2Vec succeeds but TF-IDF fails
# print("Examples where Word2Vec succeeds but TF-IDF fails:\n")
# for i in indices[:3]:  # Limit to 3 cases
#     print(f"Review: {X_test.iloc[i]}")
#     print(f"Actual Label: {y_test.iloc[i]}")
#     print(f"TF-IDF Prediction: {y_pred1[i]}")
#     print(f"Word2Vec Prediction: {y_pred[i]}")
#     print("-" * 80)

# Identify cases where TF-IDF fails but Word2Vec succeeds
incorrect_tfidf = (y_pred1 != y_test)  # TF-IDF misclassified
correct_w2v = (y_pred == y_test)  # Word2Vec correctly classified

# Find indices where TF-IDF fails and Word2Vec succeeds
indices = np.where(incorrect_tfidf & correct_w2v)[0]


def is_valid_review(text, min_words=50, max_words=250):
    return min_words <= len(text.split()) <= max_words 

# Filter cases based on word count and <br> removal
filtered_indices = [i for i in indices if is_valid_review(X_test.iloc[i])]

# Print up to 3 cases where Word2Vec succeeds but TF-IDF fails
print("Examples where Word2Vec succeeds but TF-IDF fails:\n")
for i in filtered_indices[:3]:  # Limit to 3 cases
    print(f"Review: {X_test.iloc[i]}")
    print(f"Actual Label: {y_test.iloc[i]}")
    print(f"TF-IDF Prediction: {y_pred1[i]}")
    print(f"Word2Vec Prediction: {y_pred[i]}")
    print("-" * 80)

# # If no cases are found, print a message
# if len(filtered_indices) == 0:
#     print("No valid cases found where Word2Vec succeeds but TF-IDF fails.")




Examples where Word2Vec succeeds but TF-IDF fails:

Review: The thing I remember most about this film is that it used to air on local KTLA TV (Ch. 5) during every Christmas season during the mid to late 70s, mainly due to the fact that the true story took place on or near Christmas Eve. It was always a bit disturbing to see the hell that this girl goes through, being the lone survivor of a plane crash in the Peruvian jungle. The graphic scene of this young girl pulling leeches out of her infected leg made quite an impression on this young viewer. Not quite the kind of Christmas cheer I was used to seeing at the time. Definitely not a Rankin-Bass production.
Actual Label: positive
TF-IDF Prediction: negative
Word2Vec Prediction: positive
--------------------------------------------------------------------------------
Review: I was so disappointed in this movie. I don't know much about the true story, so I was eager to see it play out on film and educate myself about a little slice of hi

In [13]:
# Identify cases where Word2Vec misclassifies
incorrect_w2v = (y_pred != y_test)  # Word2Vec misclassified

# Find indices where Word2Vec fails
indices = np.where(incorrect_w2v)[0]

# Define function to check word count range and filter out '<br />'
def is_valid_review(text, min_words=50, max_words=250):
    return min_words <= len(text.split()) <= max_words 

# Filter cases based on word count and <br> removal
filtered_indices = [i for i in indices if is_valid_review(X_test.iloc[i])]

# Print up to 3 such cases
print("Examples where Word2Vec fails:\n")
for i in filtered_indices[:3]:  # Limit to 3 cases
    print(f"Review: {X_test.iloc[i]}")
    print(f"Actual Label: {y_test.iloc[i]}")
    print(f"Word2Vec Prediction: {y_pred[i]}")
    print("-" * 80)


Examples where Word2Vec fails:

Review: I really liked this Summerslam due to the look of the arena, the curtains and just the look overall was interesting to me for some reason. Anyways, this could have been one of the best Summerslam's ever if the WWF didn't have Lex Luger in the main event against Yokozuna, now for it's time it was ok to have a huge fat man vs a strong man but I'm glad times have changed. It was a terrible main event just like every match Luger is in is terrible. Other matches on the card were Razor Ramon vs Ted Dibiase, Steiner Brothers vs Heavenly Bodies, Shawn Michaels vs Curt Hening, this was the event where Shawn named his big monster of a body guard Diesel, IRS vs 1-2-3 Kid, Bret Hart first takes on Doink then takes on Jerry Lawler and stuff with the Harts and Lawler was always very interesting, then Ludvig Borga destroyed Marty Jannetty, Undertaker took on Giant Gonzalez in another terrible match, The Smoking Gunns and Tatanka took on Bam Bam Bigelow and the 

In [9]:
# Get misclassified examples
misclassified_df = pd.DataFrame({
    'review': X_test[y_test != y_pred],
    'true_sentiment': y_test[y_test != y_pred],
    'predicted_sentiment': y_pred[y_test != y_pred],
    'probability': np.max(lr_w2v.predict_proba(X_test_w2v)[y_test != y_pred], axis=1)
})

# Add additional features
misclassified_df['review_length'] = misclassified_df['review'].apply(len)
misclassified_df['word_count'] = misclassified_df['review'].apply(lambda x: len(x.split()))

# Save to CSV
misclassified_df.to_csv('misclassified_w2v_lr.csv', index=False)

In [10]:
rf_w2v = RandomForestClassifier(n_estimators=100, random_state=42)
rf_w2v.fit(X_train_w2v, y_train)

y_pred = rf_w2v.predict(X_test_w2v)
print("Random Forest with Word2Vec Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Random Forest with Word2Vec Accuracy: 0.7972
              precision    recall  f1-score   support

    negative       0.79      0.80      0.80      4961
    positive       0.80      0.79      0.80      5039

    accuracy                           0.80     10000
   macro avg       0.80      0.80      0.80     10000
weighted avg       0.80      0.80      0.80     10000



In [11]:
svm_w2v = SVC(kernel='linear', probability=True)
svm_w2v.fit(X_train_w2v, y_train)

y_pred = svm_w2v.predict(X_test_w2v)
print("SVM with Word2Vec Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

SVM with Word2Vec Accuracy: 0.8719
              precision    recall  f1-score   support

    negative       0.88      0.86      0.87      4961
    positive       0.87      0.88      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



WORD2VEC CBOW

In [15]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

# Tokenize reviews
tokenized_reviews = [word_tokenize(review.lower()) for review in X_train]

# Train Word2Vec model
w2v_cbow_model = Word2Vec(tokenized_reviews,
                     sg=0,
                    vector_size=300,  # embedding dimension
                    window=5,         # context window size
                    min_count=5,      # minimum word frequency
                    workers=4,        # parallel threads
                    epochs=10)       # training iterations

# Save the model
w2v_cbow_model.save("cbow_imdb.model")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [16]:
import numpy as np

def average_word_vectors(review, model, vocabulary, num_features):
    feature_vector = np.zeros((num_features,), dtype="float64")
    nwords = 0
    
    for word in word_tokenize(review.lower()):
        if word in vocabulary:
            feature_vector = np.add(feature_vector, model.wv[word])
            nwords += 1
    
    if nwords:
        feature_vector = np.divide(feature_vector, nwords)
        
    return feature_vector

# Get vocabulary
vocabulary = set(w2v_cbow_model.wv.index_to_key)

# Create averaged feature vectors
X_train_w2v_cbow = np.array([average_word_vectors(review, w2v_cbow_model, vocabulary, 300) 
                       for review in X_train])
X_test_w2v_cbow = np.array([average_word_vectors(review, w2v_cbow_model, vocabulary, 300) 
                      for review in X_test])

In [17]:
lr_w2v_cbow = LogisticRegression(max_iter=1000)
lr_w2v_cbow.fit(X_train_w2v_cbow, y_train)

y_pred = lr_w2v.predict(X_test_w2v_cbow)
print("Logistic Regression with Word2Vec Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Logistic Regression with Word2Vec Accuracy: 0.5999
              precision    recall  f1-score   support

    negative       0.56      0.96      0.70      4961
    positive       0.87      0.24      0.38      5039

    accuracy                           0.60     10000
   macro avg       0.71      0.60      0.54     10000
weighted avg       0.71      0.60      0.54     10000

